# File B: Two-LLM Disagreement Pipeline

Ministral and Llama 70B both annotate independently.
Where they agree: use that label.
Where they disagree: send to qwen3.5-27b as adjudicator (sees both labels + explanations).
Then evaluate against gold standard and compare with File A.

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import os
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, cohen_kappa_score, confusion_matrix
)
import importlib
import annotation_setup
importlib.reload(annotation_setup)
from annotation_setup import VALID_LABELS, instructions, reminder, parse_label, parse_explanation

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST            = "https://maki.uni-mannheim.de/v1"
MODEL_A         = "ministral-3-14b"
MODEL_B         = "Meta-Llama-3.3-70B-Instruct-AWQ-INT4"
MODEL_TIEBREAK  = "qwen3.5-27b"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Model A:    {MODEL_A}")
print(f"Model B:    {MODEL_B}")
print(f"Tiebreaker: {MODEL_TIEBREAK}")

Model A:    ministral-3-14b
Model B:    Meta-Llama-3.3-70B-Instruct-AWQ-INT4
Tiebreaker: qwen3.5-27b


In [7]:
# paste your current FEW_SHOT_EXAMPLES list here
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME — integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG — direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG — institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME — group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # 5. NO_CRIME_FRAME — international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # 6. CRIME_FRAME_NEG — group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS — successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS — social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS — prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME — integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135 — generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30 — explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189 — political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 13
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. CRIME_FRAME_NEG
  7. CRIME_FRAME_POS
  8. CRIME_FRAME_POS
  9. CRIME_FRAME_POS
  10. NO_CRIME_FRAME
  11. CRIME_FRAME_NEG
  12. CRIME_FRAME_NEG
  13. NO_CRIME_FRAME


In [8]:
def call_api(messages, model, temperature=0.0):
    payload = {
        "model": model,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload, headers=headers, timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def format_few_shot_block(examples):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def annotate(text, model, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, model, temperature)


def adjudicate(text, label_a, expl_a, label_b, expl_b):
    system = f"""Sie sind ein erfahrener Annotationsexperte fuer Inhaltsanalyse.

Zwei unabhaengige Modelle haben fuer den folgenden Absatz unterschiedliche Labels vergeben:

Modell A: {label_a}
Begruendung A: {expl_a}

Modell B: {label_b}
Begruendung B: {expl_b}

Ihre Aufgabe ist es, das korrekte Label zu bestimmen. Die drei moeglichen Labels sind:

CRIME_FRAME_NEG: Die Gruppe wird explizit als Quelle von Bedrohung, Kriminalitaet oder Sicherheitsproblem im deutschen Inland dargestellt.
CRIME_FRAME_POS: Eine Sicherheitsmassnahme wird explizit als Erfolg gerahmt, oder Praevention/Differenzierung steht im Vordergrund.
NO_CRIME_FRAME: Alles andere. Insbesondere wenn die Gruppe Opfer ist, die Gewalt ausserhalb Deutschlands stattfindet, oder kein expliziter Sicherheitsrahmen vorliegt.

Lesen Sie beide Begruendungen sorgfaeltig. Entscheiden Sie welche Begruendung besser auf den Text passt und warum."""

    prompt = (
        f"Hier ist der Absatz:\n\n{text}\n\n"
        "Antworten Sie in diesem Format:\n"
        "Reasoning: <ein Satz warum Sie sich fuer dieses Label entschieden haben>\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>"
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, MODEL_TIEBREAK, temperature=0.0)


def parse_reasoning(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("reasoning:"):
            return line[len("reasoning:"):].strip()
    return ""


print("Functions loaded.")

Functions loaded.


In [9]:
human_completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed.csv")
human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Rows: {len(human_completed)}")
print(human_completed["final_human_label"].value_counts())

Rows: 250
final_human_label
NO_CRIME_FRAME     129
CRIME_FRAME_NEG    114
CRIME_FRAME_POS      7
Name: count, dtype: int64


## Step 1: Annotate with Model A (Ministral)

In [10]:
OUTPUT_A = RESULTS_DIR / "two_llm_model_a.csv"

if OUTPUT_A.exists():
    done_a   = pd.read_csv(OUTPUT_A)
    done_ids = set(done_a["testset_id"])
    print(f"Resuming: {len(done_a)} done")
else:
    done_a   = pd.DataFrame()
    done_ids = set()
    print(f"Starting: {len(human_completed)} rows")

buffer = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue
    raw   = annotate(str(row["text_block_en"]), MODEL_A)
    buffer.append({
        "testset_id":        row["testset_id"],
        "group":             row["group"],
        "text_block_en":     row["text_block_en"],
        "final_human_label": row["final_human_label"],
        "label_a":           parse_label(raw),
        "explanation_a":     parse_explanation(raw),
    })
    done_ids.add(row["testset_id"])
    if len(buffer) % 50 == 0:
        chunk  = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_A, mode="a", header=not OUTPUT_A.exists(), index=False)
        done_a = pd.concat([done_a, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_a)}/{len(human_completed)}]")

if buffer:
    chunk  = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_A, mode="a", header=not OUTPUT_A.exists(), index=False)
    done_a = pd.concat([done_a, chunk], ignore_index=True)

print(f"\nModel A done. {len(done_a)} rows.")
print(done_a["label_a"].value_counts())

Starting: 250 rows
  [50/250]
  [100/250]
  [150/250]
  [200/250]
  [250/250]

Model A done. 250 rows.
label_a
NO_CRIME_FRAME     145
CRIME_FRAME_NEG    104
CRIME_FRAME_POS      1
Name: count, dtype: int64


## Step 2: Annotate with Model B (Llama 70B)

In [11]:
OUTPUT_B = RESULTS_DIR / "two_llm_model_b.csv"

if OUTPUT_B.exists():
    done_b   = pd.read_csv(OUTPUT_B)
    done_ids = set(done_b["testset_id"])
    print(f"Resuming: {len(done_b)} done")
else:
    done_b   = pd.DataFrame()
    done_ids = set()
    print(f"Starting: {len(human_completed)} rows")

buffer = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids:
        continue
    raw   = annotate(str(row["text_block_en"]), MODEL_B)
    buffer.append({
        "testset_id":  row["testset_id"],
        "label_b":     parse_label(raw),
        "explanation_b": parse_explanation(raw),
    })
    done_ids.add(row["testset_id"])
    if len(buffer) % 50 == 0:
        chunk  = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
        done_b = pd.concat([done_b, chunk], ignore_index=True)
        buffer = []
        print(f"  [{len(done_b)}/{len(human_completed)}]")

if buffer:
    chunk  = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
    done_b = pd.concat([done_b, chunk], ignore_index=True)

print(f"\nModel B done. {len(done_b)} rows.")
print(done_b["label_b"].value_counts())

Starting: 250 rows
  [50/250]
  [100/250]
  [150/250]
  [200/250]
  [250/250]

Model B done. 250 rows.
label_b
CRIME_FRAME_NEG    162
NO_CRIME_FRAME      86
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Step 3: Find Conflicts and Adjudicate with Qwen

In [12]:
done_a = pd.read_csv(OUTPUT_A)
done_b = pd.read_csv(OUTPUT_B)

merged = done_a.merge(done_b[["testset_id", "label_b", "explanation_b"]], on="testset_id", how="left")

conflicts  = merged[merged["label_a"] != merged["label_b"]].copy()
agreements = merged[merged["label_a"] == merged["label_b"]].copy()

print(f"Total:      {len(merged)}")
print(f"Agreements: {len(agreements)} ({len(agreements)/len(merged)*100:.1f}%)")
print(f"Conflicts:  {len(conflicts)} ({len(conflicts)/len(merged)*100:.1f}%)")
print("\nConflict breakdown:")
print(conflicts.groupby(["label_a", "label_b"]).size().reset_index(name="count").to_string(index=False))

Total:      250
Agreements: 176 (70.4%)
Conflicts:  74 (29.6%)

Conflict breakdown:
        label_a         label_b  count
CRIME_FRAME_NEG CRIME_FRAME_POS      1
CRIME_FRAME_NEG  NO_CRIME_FRAME      7
 NO_CRIME_FRAME CRIME_FRAME_NEG     66


In [13]:
OUTPUT_TIEBREAK = RESULTS_DIR / "two_llm_tiebreak.csv"

if OUTPUT_TIEBREAK.exists():
    done_tb   = pd.read_csv(OUTPUT_TIEBREAK)
    done_ids  = set(done_tb["testset_id"])
    print(f"Resuming tiebreak: {len(done_tb)} done, {len(conflicts) - len(done_tb)} remaining")
else:
    done_tb  = pd.DataFrame()
    done_ids = set()
    print(f"Adjudicating {len(conflicts)} conflicts with {MODEL_TIEBREAK}")

buffer = []

for i, row in conflicts.iterrows():
    if row["testset_id"] in done_ids:
        continue
    raw      = adjudicate(
        str(row["text_block_en"]),
        row["label_a"], str(row["explanation_a"]),
        row["label_b"], str(row["explanation_b"])
    )
    label_tb = parse_label(raw)
    reason   = parse_reasoning(raw)
    buffer.append({
        "testset_id":     row["testset_id"],
        "label_tiebreak": label_tb,
        "tiebreak_reasoning": reason,
    })
    done_ids.add(row["testset_id"])
    if len(buffer) % 20 == 0:
        chunk   = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_TIEBREAK, mode="a", header=not OUTPUT_TIEBREAK.exists(), index=False)
        done_tb = pd.concat([done_tb, chunk], ignore_index=True)
        buffer  = []
        print(f"  [{len(done_tb)}/{len(conflicts)}] adjudicated")

if buffer:
    chunk   = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_TIEBREAK, mode="a", header=not OUTPUT_TIEBREAK.exists(), index=False)
    done_tb = pd.concat([done_tb, chunk], ignore_index=True)

print(f"\nTiebreak done. {len(done_tb)} conflicts resolved.")
print(done_tb["label_tiebreak"].value_counts())

Adjudicating 74 conflicts with qwen3.5-27b
  [20/74] adjudicated
  [40/74] adjudicated
  [60/74] adjudicated

Tiebreak done. 74 conflicts resolved.
label_tiebreak
NO_CRIME_FRAME     64
CRIME_FRAME_NEG     9
CRIME_FRAME_POS     1
Name: count, dtype: int64


## Step 4: Merge and Assign Final Labels

In [14]:
done_a  = pd.read_csv(OUTPUT_A)
done_b  = pd.read_csv(OUTPUT_B)
done_tb = pd.read_csv(OUTPUT_TIEBREAK)

merged = done_a.merge(done_b[["testset_id", "label_b", "explanation_b"]], on="testset_id", how="left")
merged = merged.merge(done_tb[["testset_id", "label_tiebreak", "tiebreak_reasoning"]], on="testset_id", how="left")

def assign_final(row):
    if row["label_a"] == row["label_b"]:
        return row["label_a"], "agreement"
    elif pd.notna(row.get("label_tiebreak")) and row["label_tiebreak"] in VALID_LABELS:
        return row["label_tiebreak"], "tiebreak"
    else:
        return row["label_a"], "fallback_to_a"

results = merged.apply(assign_final, axis=1, result_type="expand")
merged["final_label"]  = results[0]
merged["decision_type"]= results[1]

merged.to_csv(RESULTS_DIR / "two_llm_final.csv", index=False)

print(f"Total: {len(merged)}")
print(merged["decision_type"].value_counts())
print()
print("Final label distribution:")
print(merged["final_label"].value_counts())

Total: 250
decision_type
agreement    176
tiebreak      74
Name: count, dtype: int64

Final label distribution:
final_label
NO_CRIME_FRAME     143
CRIME_FRAME_NEG    105
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Step 5: Evaluate

In [15]:
def evaluate(y_true, y_pred, name):
    valid = [(t, p) for t, p in zip(y_true, y_pred) if t in VALID_LABELS and p in VALID_LABELS]
    if not valid:
        print(f"{name}: no valid predictions")
        return {}
    yt, yp   = zip(*valid)
    acc      = accuracy_score(yt, yp)
    f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
    f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
    kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)
    alpha_data = np.array([[VALID_LABELS.index(t) for t in yt], [VALID_LABELS.index(p) for p in yp]])
    try:
        alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")
    except:
        alpha = float("nan")
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print(f"Accuracy: {acc:.3f} | Macro F1: {f1_macro:.3f} | Weighted F1: {f1_w:.3f}")
    print(f"Kappa: {kappa:.3f} | Alpha: {alpha:.3f}")
    print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(yt, yp, labels=VALID_LABELS))
    return {"name": name, "accuracy": acc, "macro_f1": f1_macro, "weighted_f1": f1_w, "kappa": kappa, "alpha": alpha}


merged = pd.read_csv(RESULTS_DIR / "two_llm_final.csv")

res_final = evaluate(
    merged["final_human_label"].tolist(),
    merged["final_label"].tolist(),
    "Two-LLM Pipeline (Ministral + Llama + Qwen tiebreak)"
)

res_a = evaluate(
    merged["final_human_label"].tolist(),
    merged["label_a"].tolist(),
    "Model A only (Ministral)"
)

res_b = evaluate(
    merged["final_human_label"].tolist(),
    merged["label_b"].tolist(),
    "Model B only (Llama 70B)"
)

print("\n=== Summary ===")
for r in [res_a, res_b, res_final]:
    print(f"{r['name']}: kappa={r['kappa']:.3f} macro_f1={r['macro_f1']:.3f} weighted_f1={r['weighted_f1']:.3f}")


Two-LLM Pipeline (Ministral + Llama + Qwen tiebreak)
Accuracy: 0.800 | Macro F1: 0.684 | Weighted F1: 0.796
Kappa: 0.610 | Alpha: 0.610
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.78      0.87      0.82       129
CRIME_FRAME_NEG       0.82      0.75      0.79       114
CRIME_FRAME_POS       1.00      0.29      0.44         7

       accuracy                           0.80       250
      macro avg       0.87      0.64      0.68       250
   weighted avg       0.81      0.80      0.80       250

Confusion matrix:
[[112  17   0]
 [ 28  86   0]
 [  3   2   2]]

Model A only (Ministral)
Accuracy: 0.792 | Macro F1: 0.615 | Weighted F1: 0.784
Kappa: 0.593 | Alpha: 0.592
                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.78      0.88      0.82       129
CRIME_FRAME_NEG       0.81      0.74      0.77       114
CRIME_FRAME_POS       1.00      0.14      0.25         7

       accuracy                           0.79       250
 

In [16]:
# how well did qwen adjudicate?
tiebreak_rows = merged[merged["decision_type"] == "tiebreak"].copy()
tiebreak_rows["correct"] = tiebreak_rows["final_human_label"] == tiebreak_rows["final_label"]
print(f"Tiebreak accuracy: {tiebreak_rows['correct'].mean():.3f} ({len(tiebreak_rows)} conflict rows)")
print()
print("Tiebreak sided with:")
print(f"  Model A: {(tiebreak_rows['final_label'] == tiebreak_rows['label_a']).sum()}")
print(f"  Model B: {(tiebreak_rows['final_label'] == tiebreak_rows['label_b']).sum()}")
print(f"  Neither: {((tiebreak_rows['final_label'] != tiebreak_rows['label_a']) & (tiebreak_rows['final_label'] != tiebreak_rows['label_b'])).sum()}")

Tiebreak accuracy: 0.649 (74 conflict rows)

Tiebreak sided with:
  Model A: 61
  Model B: 13
  Neither: 0
